# 07 · NDVI over time: trend and change detection

One NDVI snapshot says where the field is. A series of dates says where it is
going. This notebook builds observations from a CSV of per-date statistics,
computes the trend, and reports change between the first and last date.

In [ ]:
%matplotlib inline
# --- Vibes Lite bootstrap -----------------------------------------------------
# These notebooks drive the processing engines directly (no HTTP layer needed).
import sys
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import config, data, geospatial, preprocessing, indices, analysis
import opencv as vl_cv          # Vibes Lite's own engine (it wraps cv2 internally)
import segmentation, timeseries, visualization, storage

PATHS = config.ensure_directories()
NOTEBOOK_OUT = PATHS["outputs"] / "notebooks"
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("raw data     :", PATHS["raw_data"].relative_to(ROOT))
print("notebook out :", NOTEBOOK_OUT.relative_to(ROOT))

In [ ]:
DEMO_NAME = "synthetic_fields"


def demo_scene(name: str = DEMO_NAME, size: int = 900):
    """Return (raster, aoi, csv) for the labelled synthetic demo scene.

    The scene is only generated when missing. It is explicitly synthetic - its
    GeoTIFF tags carry SYNTHETIC=true - so nothing here pretends to be a real
    satellite acquisition.
    """
    raster = PATHS["raw_data"] / f"{name}.tif"
    aoi = PATHS["raw_data"] / f"{name}.geojson"
    table = PATHS["raw_data"] / f"{name}_ndvi_history.csv"
    if not raster.exists() or not aoi.exists():
        scene = data.generate_synthetic_scene(name=name, size=size, overwrite=True)
        return scene["paths"]["raster"], scene["paths"]["aoi"], scene["paths"]["table"]
    return raster, aoi, table


RASTER, AOI_FILE, HISTORY_CSV = demo_scene()
print("raster :", RASTER.name)
print("aoi    :", AOI_FILE.name)
print("history:", HISTORY_CSV.name)

## 1. Observations from a CSV

The demo CSV has `date`, `field`, `ndvi_mean`, `cloud_cover_pct` columns - one row per field per date.

In [ ]:
history = pd.read_csv(HISTORY_CSV)
print("rows:", len(history), "| columns:", list(history.columns))
history.head(7)

The CSV is the convenient path; per-field observations can also be built inline, straight from stored analysis runs.

In [ ]:
csv_payload = timeseries.observations_from_csv(str(HISTORY_CSV), index_name="NDVI")
print("rows          :", csv_payload["rows"])
print("group column  :", csv_payload["group_column"])
print("observations  :", len(csv_payload["observations"]))
print(json.dumps(csv_payload["observations"][0], indent=2, default=str))

## 2. Trend and change detection

`analyze_timeseries` fits a least-squares trend, classifies the change between the first and last observation, and tracks the healthy/weak share.

In [ ]:
payload = timeseries.analyze_timeseries(csv_payload["observations"], index_name="NDVI", change_threshold=0.05)
print("payload keys :", sorted(payload))
print("series keys  :", sorted(payload["series"]))
print("trend        :", json.dumps(payload["trend"], indent=2, default=str))
print("\nchange       :", json.dumps(payload["change"], indent=2, default=str))
print("\nhealth trend :", json.dumps(payload["health_trend"], indent=2, default=str))

In [ ]:
frame = pd.DataFrame(payload["series"]["observations"])
frame["date"] = pd.to_datetime(frame["date"])
frame[["date", "mean", "median", "healthy_pct", "weak_pct"]].round(4)

In [ ]:
figure, axes = plt.subplots(figsize=(9, 4.5))
axes.plot(frame["date"], frame["mean"], marker="o", color="#1f6f4a", label="mean NDVI")
axes.axhline(y=0.2, color="#8c3a1f", linestyle="--", linewidth=1, label="very low (< 0.20)")
axes.axhline(y=0.6, color="#2e7d32", linestyle="--", linewidth=1, label="healthy (> 0.60)")
axes.set_title("NDVI trajectory - " + str(payload["series"].get("index", "NDVI")) + " (" + str(payload["series"]["count"]) + " observations)")
axes.set_ylabel("NDVI")
axes.grid(alpha=0.25)
axes.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. The chart the dashboard shows

Same numbers, rendered by the visualization engine into `outputs/`.

In [ ]:
chart_path = visualization.save_timeseries_chart(
    payload["series"],
    NOTEBOOK_OUT / "07_ndvi_timeseries.png",
    trend=payload["trend"],
    index_name="NDVI",
    title="NDVI over time (synthetic demo)",
)
print("saved:", pathlib.Path(chart_path).relative_to(ROOT))
print("\nsummary:", json.dumps(payload.get("summary", {}), indent=2, default=str)[:600])

## 4. Adding real observations

Observations can also come from **stored analysis results** - run NDVI on several dates of the same field, then feed the result ids back in:

In [ ]:
stored = storage.list_results(kind="ndvi", limit=5)
print("stored NDVI results:", [record["id"] for record in stored])
series_from_results = timeseries.series_from_result_records(stored, index_name="NDVI")
print("observations from stored results:", series_from_results.get("count", 0))
print("\nA real workflow is: ingest one Sentinel-2 scene per date -> POST /api/ndvi for each -> "
      "POST /api/timeseries with the resulting ids.")

**Change detection in practice.** A rising NDVI trend across a season is
expected for an establishing crop; what matters is a *break* in that pattern -
a drop against the previous date, or one field falling behind its neighbours.
Cloud cover matters: check the `cloud_cover_pct` column before trusting a dip.